### 1. Setup and Environment
This section will mount Google Drive and install dependencies.
CSE 6363 Project-Sentiment Analysis With BERT-based-uncased

In [ ]:
from google.colab import drive
import os

print("Mounting Google Drive...")
drive.mount('/content/drive')
print("Drive mounted successfully.")

project_path = "/content/drive/My Drive/CSE 6363 Project-Sentiment Analysis With BERT-based-uncased/Sentiment Analysis"

print(f"Changing directory to: {project_path}")
os.chdir(project_path)


Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted successfully.
Changing directory to: /content/drive/My Drive/CSE 6363 Project-Sentiment Analysis With BERT-based-uncased/Sentiment Analysis


# Statistical Model Validation

In this notebook, we perform rigorous statistical testing to confirm that our results are significant and not due to random chance.

We will perform two key tests:
1.  **McNemar's Test:** To statistically compare our best BERT model against the CNN Baseline. This answers the question: *"Is the difference in accuracy real?"*
2.  **Bootstrap Confidence Intervals:** To estimate the true accuracy range of our best model with 95% confidence.

## 1. McNemar's Test (Baseline vs. BERT)

We compare the predictions of `cnn_baseline` (Model A) against `bert_full_finetune_seed123` (Model B).
* **Null Hypothesis ($H_0$):** The two models have the same error rate.
* **Alternative Hypothesis ($H_1$):** The error rates are different.
* **Significance Level ($\alpha$):** 0.05

In [9]:
# Updated command: removed the "../" prefix
!python src/postprocessing/validations.py \
    --task mcnemar \
    --run_a "cnn_baseline.pt" \
    --run_b "bert_full_finetune_seed123.pt"


--- Running McNemar's Test ---
Model A (Baseline): cnn_baseline.pt
Model B (Challenger): bert_full_finetune_seed123.pt

Contingency Table:
Both Correct: 21235
cnn_baseline.pt Correct, bert_full_finetune_seed123.pt Wrong: 588
cnn_baseline.pt Wrong,   bert_full_finetune_seed123.pt Correct: 2327
Both Wrong:   850

McNemar's Statistic (Chi2): 1036.242
P-Value: 2.3812595604e-227

>> RESULT: Statistically Significant Difference (p < 0.05)
>> CONCLUSION: bert_full_finetune_seed123.pt is significantly BETTER than cnn_baseline.pt.


## 2. Bootstrap Confidence Intervals

Standard accuracy metrics give a single point estimate (e.g., 94.24%). Bootstrapping allows us to quantify the uncertainty around this number by resampling the test set 1,000 times.

This gives us a range (e.g., 93.9% - 94.5%) where the "true" model accuracy likely lies.

In [10]:
!python src/postprocessing/validations.py \
    --task bootstrap \
    --run_name "bert_full_finetune_seed123.pt"


--- Running Bootstrap Confidence Interval (n=1000) ---
Model: bert_full_finetune_seed123.pt
Resampling test set...
100% 1000/1000 [00:01<00:00, 940.53it/s]

Mean Accuracy: 94.25%
95% Confidence Interval: [93.96%, 94.53%]
Margin of Error: +/- 0.29%


## 3. Seed Variance Analysis

To validate the stability of our layer-freezing strategy, we trained each model configuration across three random seeds (42, 123, 2025).

**Results (F1 Score):**
* **Full Fine-Tuning:** 93.61%, 94.29%, 93.98% -> **Mean: 93.96% ± 0.34%**
* **Frozen-8:** 93.35%, 93.65%, 93.53% -> **Mean: 93.51% ± 0.15%**
* **Frozen-11:** 92.09%, 92.28%, 92.21% -> **Mean: 92.19% ± 0.09%**

**Conclusion:**
The standard deviation decreases as we freeze more layers (0.34% -> 0.15% -> 0.09%). This confirms our hypothesis that layer freezing acts as a **regularizer**, making the model's performance significantly more stable and reproducible across different random initializations.